# Telco Customer Churn Analytics
## 03 - Business Insights

This notebook converts exploratory analysis into
business-oriented findings.

### Objectives

- Quantify major churn patterns
- Compare customer segments
- Identify high-churn segments
- Analyze revenue exposure
- Identify customer groups requiring attention
- Create a final executive summary

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../data/cleaned/telco_churn_cleaned.csv")

In [3]:
df.shape

(7043, 25)

In [4]:
tenure_order = [
    "0-12 Months",
    "13-24 Months",
    "25-48 Months",
    "49-60 Months",
    "60+ Months"
]

charge_order = [
    "Under $30",
    "$30-$59",
    "$60-$89",
    "$90+"
]

df["Tenure_Group"] = pd.Categorical(
    df["Tenure_Group"],
    categories=tenure_order,
    ordered=True
)

df["Monthly_Charges_Group"] = pd.Categorical(
    df["Monthly_Charges_Group"],
    categories=charge_order,
    ordered=True
)

In [5]:
total_customers = df["customerID"].nunique()

churned_customers = df["Churn_Flag"].sum()

retained_customers = total_customers - churned_customers

churn_rate = df["Churn_Flag"].mean() * 100

average_tenure = df["tenure"].mean()

average_monthly_charges = df["MonthlyCharges"].mean()

total_monthly_charges = df["MonthlyCharges"].sum()

average_total_charges = df["TotalCharges"].mean()

total_historical_charges = df["TotalCharges"].sum()

In [6]:
kpi_summary = pd.DataFrame({
    "Metric": [
        "Total Customers",
        "Churned Customers",
        "Retained Customers",
        "Overall Churn Rate (%)",
        "Average Tenure (Months)",
        "Average Monthly Charges",
        "Total Monthly Charges",
        "Average Total Charges",
        "Total Historical Charges"
    ],
    "Value": [
        total_customers,
        churned_customers,
        retained_customers,
        round(churn_rate, 2),
        round(average_tenure, 2),
        round(average_monthly_charges, 2),
        round(total_monthly_charges, 2),
        round(average_total_charges, 2),
        round(total_historical_charges, 2)
    ]
})

kpi_summary

,Metric,Value
0,Total Customers,7043.00
1,Churned Customers,1869.00
2,Retained Customers,5174.00
3,Overall Churn Rate (%),26.54
4,Average Tenure (Months),32.37
5,Average Monthly Charges,64.76
6,Total Monthly Charges,456116.60
7,Average Total Charges,2279.73
8,Total Historical Charges,16056168.70


In [7]:
contract_analysis = (
    df.groupby("Contract", observed=True)
    .agg(
        Customers=("customerID", "nunique"),
        Churned_Customers=("Churn_Flag", "sum"),
        Churn_Rate=("Churn_Flag", "mean")
    )
    .reset_index()
)

contract_analysis["Churn_Rate"] *= 100

contract_analysis

,Contract,Customers,Churned_Customers,Churn_Rate
0,Month-to-month,3875,1655,42.709677
1,One year,1473,166,11.269518
2,Two year,1695,48,2.831858


In [8]:
contract_analysis.sort_values(
    "Churn_Rate",
    ascending=False
)

,Contract,Customers,Churned_Customers,Churn_Rate
0,Month-to-month,3875,1655,42.709677
1,One year,1473,166,11.269518
2,Two year,1695,48,2.831858


In [9]:
highest_contract = contract_analysis.loc[
    contract_analysis["Churn_Rate"].idxmax()
]

highest_contract

Contract             Month-to-month
Customers                      3875
Churned_Customers              1655
Churn_Rate                42.709677
Name: 0, dtype: object

In [10]:
print(
    "Highest observed churn-rate contract segment:",
    highest_contract["Contract"]
)

print(
    "Churn rate:",
    round(highest_contract["Churn_Rate"], 2),
    "%"
)

Highest observed churn-rate contract segment: Month-to-month
Churn rate: 42.71 %


In [11]:
tenure_analysis = (
    df.groupby("Tenure_Group", observed=True)
    .agg(
        Customers=("customerID", "nunique"),
        Churned_Customers=("Churn_Flag", "sum"),
        Churn_Rate=("Churn_Flag", "mean")
    )
    .reset_index()
)

tenure_analysis["Churn_Rate"] *= 100

tenure_analysis

,Tenure_Group,Customers,Churned_Customers,Churn_Rate
0,0-12 Months,2186,1037,47.438243
1,13-24 Months,1024,294,28.710938
2,25-48 Months,1594,325,20.388959
3,49-60 Months,832,120,14.423077
4,60+ Months,1407,93,6.609808


In [12]:
highest_tenure = tenure_analysis.loc[
    tenure_analysis["Churn_Rate"].idxmax()
]

highest_tenure

Tenure_Group         0-12 Months
Customers                   2186
Churned_Customers           1037
Churn_Rate             47.438243
Name: 0, dtype: object

In [13]:
lowest_tenure = tenure_analysis.loc[
    tenure_analysis["Churn_Rate"].idxmin()
]

lowest_tenure

Tenure_Group         60+ Months
Customers                  1407
Churned_Customers            93
Churn_Rate             6.609808
Name: 4, dtype: object

In [14]:
early_tenure = df[df["tenure"] <= 12]

early_tenure_churn_rate = (
    early_tenure["Churn_Flag"].mean() * 100
)

early_tenure_customers = len(early_tenure)

early_tenure_churned = early_tenure["Churn_Flag"].sum()

print("0-12 Month Customers:", early_tenure_customers)
print("Churned Customers:", early_tenure_churned)
print(
    "Churn Rate:",
    round(early_tenure_churn_rate, 2),
    "%"
)

0-12 Month Customers: 2186
Churned Customers: 1037
Churn Rate: 47.44 %


In [15]:
charge_analysis = (
    df.groupby("Monthly_Charges_Group", observed=True)
    .agg(
        Customers=("customerID", "nunique"),
        Churned_Customers=("Churn_Flag", "sum"),
        Churn_Rate=("Churn_Flag", "mean"),
        Avg_Monthly_Charges=("MonthlyCharges", "mean")
    )
    .reset_index()
)

charge_analysis["Churn_Rate"] *= 100

charge_analysis

,Monthly_Charges_Group,Customers,Churned_Customers,Churn_Rate,Avg_Monthly_Charges
0,Under $30,1653,162,9.800363,21.503539
1,$30-$59,1254,327,26.076555,48.588198
2,$60-$89,2392,807,33.737458,76.722471
3,$90+,1744,573,32.855505,100.987099


In [16]:
highest_charge_segment = charge_analysis.loc[
    charge_analysis["Churn_Rate"].idxmax()
]

highest_charge_segment

Monthly_Charges_Group      $60-$89
Customers                     2392
Churned_Customers              807
Churn_Rate               33.737458
Avg_Monthly_Charges      76.722471
Name: 2, dtype: object

In [17]:
internet_analysis = (
    df.groupby("InternetService", observed=True)
    .agg(
        Customers=("customerID", "nunique"),
        Churned_Customers=("Churn_Flag", "sum"),
        Churn_Rate=("Churn_Flag", "mean")
    )
    .reset_index()
)

internet_analysis["Churn_Rate"] *= 100

internet_analysis

,InternetService,Customers,Churned_Customers,Churn_Rate
0,DSL,2421,459,18.959108
1,Fiber optic,3096,1297,41.892765
2,No,1526,113,7.404980


In [18]:
highest_internet = internet_analysis.loc[
    internet_analysis["Churn_Rate"].idxmax()
]

highest_internet

InternetService      Fiber optic
Customers                   3096
Churned_Customers           1297
Churn_Rate             41.892765
Name: 1, dtype: object

In [19]:
payment_analysis = (
    df.groupby("PaymentMethod")
    .agg(
        Customers=("customerID", "nunique"),
        Churned_Customers=("Churn_Flag", "sum"),
        Churn_Rate=("Churn_Flag", "mean")
    )
    .reset_index()
)

payment_analysis["Churn_Rate"] *= 100

payment_analysis.sort_values(
    "Churn_Rate",
    ascending=False
)

,PaymentMethod,Customers,Churned_Customers,Churn_Rate
2,Electronic check,2365,1071,45.285412
3,Mailed check,1612,308,19.106700
0,Bank transfer (automatic),1544,258,16.709845
1,Credit card (automatic),1522,232,15.243101


In [20]:
senior_analysis = (
    df.groupby("Senior_Citizen_Status")
    .agg(
        Customers=("customerID", "nunique"),
        Churned_Customers=("Churn_Flag", "sum"),
        Churn_Rate=("Churn_Flag", "mean")
    )
    .reset_index()
)

senior_analysis["Churn_Rate"] *= 100

senior_analysis

,Senior_Citizen_Status,Customers,Churned_Customers,Churn_Rate
0,Non-Senior Citizen,5901,1393,23.606168
1,Senior Citizen,1142,476,41.681261


In [21]:
service_columns = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport"
]

In [22]:
service_results = []

for column in service_columns:
    result = (
        df.groupby(column)["Churn_Flag"]
        .mean()
        .mul(100)
        .reset_index()
    )
    
    result["Service"] = column
    result = result.rename(
        columns={column: "Service_Status",
                 "Churn_Flag": "Churn_Rate"}
    )
    
    service_results.append(result)

service_analysis = pd.concat(
    service_results,
    ignore_index=True
)

service_analysis

,Service_Status,Churn_Rate,Service
0,No,41.766724,OnlineSecurity
1,No internet service,7.404980,OnlineSecurity
2,Yes,14.611194,OnlineSecurity
3,No,39.928756,OnlineBackup
4,No internet service,7.404980,OnlineBackup
5,Yes,21.531494,OnlineBackup
6,No,39.127625,DeviceProtection
7,No internet service,7.404980,DeviceProtection
8,Yes,22.502064,DeviceProtection
9,No,41.635474,TechSupport


In [23]:
high_charge_new_customers = df[
    (df["tenure"] <= 12) &
    (df["MonthlyCharges"] >= 90)
]

In [24]:
segment_customers = len(high_charge_new_customers)

segment_churned = high_charge_new_customers["Churn_Flag"].sum()

segment_churn_rate = (
    high_charge_new_customers["Churn_Flag"].mean() * 100
)

print("Customers:", segment_customers)
print("Churned:", segment_churned)
print("Churn Rate:", round(segment_churn_rate, 2), "%")

Customers: 219
Churned: 166
Churn Rate: 75.8 %


In [25]:
priority_segment = df[
    (df["Contract"] == "Month-to-month") &
    (df["MonthlyCharges"] >= 90) &
    (df["tenure"] <= 12)
]

In [26]:
priority_summary = pd.DataFrame({
    "Metric": [
        "Customers",
        "Churned Customers",
        "Churn Rate (%)",
        "Average Monthly Charges",
        "Average Tenure"
    ],
    "Value": [
        len(priority_segment),
        priority_segment["Churn_Flag"].sum(),
        priority_segment["Churn_Flag"].mean() * 100,
        priority_segment["MonthlyCharges"].mean(),
        priority_segment["tenure"].mean()
    ]
})

priority_summary

,Metric,Value
0,Customers,213.000000
1,Churned Customers,164.000000
2,Churn Rate (%),76.995305
3,Average Monthly Charges,96.600469
4,Average Tenure,5.943662


In [27]:
churned_df = df[df["Churn"] == "Yes"]

retained_df = df[df["Churn"] == "No"]

In [28]:
revenue_comparison = pd.DataFrame({
    "Metric": [
        "Customers",
        "Average Monthly Charges",
        "Average Total Charges",
        "Average Tenure"
    ],
    "Churned": [
        len(churned_df),
        churned_df["MonthlyCharges"].mean(),
        churned_df["TotalCharges"].mean(),
        churned_df["tenure"].mean()
    ],
    "Retained": [
        len(retained_df),
        retained_df["MonthlyCharges"].mean(),
        retained_df["TotalCharges"].mean(),
        retained_df["tenure"].mean()
    ]
})

revenue_comparison

,Metric,Churned,Retained
0,Customers,1869.000000,5174.000000
1,Average Monthly Charges,74.441332,61.265124
2,Average Total Charges,1531.796094,2549.911442
3,Average Tenure,17.979133,37.569965


In [29]:
churned_monthly_charges = churned_df["MonthlyCharges"].sum()

retained_monthly_charges = retained_df["MonthlyCharges"].sum()

print(
    "Monthly charges associated with churned customers:",
    round(churned_monthly_charges, 2)
)

print(
    "Monthly charges associated with retained customers:",
    round(retained_monthly_charges, 2)
)

Monthly charges associated with churned customers: 139130.85
Monthly charges associated with retained customers: 316985.75


In [30]:
contract_internet_analysis = (
    df.groupby(
        ["Contract", "InternetService"],
        observed=True
    )
    .agg(
        Customers=("customerID", "nunique"),
        Churned_Customers=("Churn_Flag", "sum"),
        Churn_Rate=("Churn_Flag", "mean")
    )
    .reset_index()
)

contract_internet_analysis["Churn_Rate"] *= 100

contract_internet_analysis.sort_values(
    "Churn_Rate",
    ascending=False
)

,Contract,InternetService,Customers,Churned_Customers,Churn_Rate
1,Month-to-month,Fiber optic,2128,1162,54.605263
0,Month-to-month,DSL,1223,394,32.215863
4,One year,Fiber optic,539,104,19.294991
2,Month-to-month,No,524,99,18.893130
3,One year,DSL,570,53,9.298246
7,Two year,Fiber optic,429,31,7.226107
5,One year,No,364,9,2.472527
6,Two year,DSL,628,12,1.910828
8,Two year,No,638,5,0.783699


In [31]:
contract_payment_analysis = (
    df.groupby(
        ["Contract", "PaymentMethod"]
    )
    .agg(
        Customers=("customerID", "nunique"),
        Churned_Customers=("Churn_Flag", "sum"),
        Churn_Rate=("Churn_Flag", "mean")
    )
    .reset_index()
)

contract_payment_analysis["Churn_Rate"] *= 100

contract_payment_analysis.sort_values(
    "Churn_Rate",
    ascending=False
)

,Contract,PaymentMethod,Customers,Churned_Customers,Churn_Rate
2,Month-to-month,Electronic check,1850,994,53.729730
0,Month-to-month,Bank transfer (automatic),589,201,34.125637
1,Month-to-month,Credit card (automatic),543,178,32.780847
3,Month-to-month,Mailed check,893,282,31.578947
6,One year,Electronic check,347,64,18.443804
5,One year,Credit card (automatic),398,41,10.301508
4,One year,Bank transfer (automatic),391,38,9.718670
10,Two year,Electronic check,168,13,7.738095
7,One year,Mailed check,337,23,6.824926
8,Two year,Bank transfer (automatic),564,19,3.368794


In [32]:
insights = pd.DataFrame({
    "Area": [
        "Overall Churn",
        "Contract",
        "Tenure",
        "Monthly Charges",
        "Internet Service",
        "Payment Method",
        "Early Tenure",
        "Priority Segment"
    ],
    "Finding": [
        f"Overall churn rate is {churn_rate:.2f}%.",
        f"{highest_contract['Contract']} has the highest observed churn rate among contract types ({highest_contract['Churn_Rate']:.2f}%).",
        f"{highest_tenure['Tenure_Group']} has the highest observed churn rate among tenure groups ({highest_tenure['Churn_Rate']:.2f}%).",
        f"{highest_charge_segment['Monthly_Charges_Group']} has the highest observed churn rate among charge groups ({highest_charge_segment['Churn_Rate']:.2f}%).",
        f"{highest_internet['InternetService']} has the highest observed churn rate among internet-service categories ({highest_internet['Churn_Rate']:.2f}%).",
        "Churn rates vary across payment-method categories.",
        f"Customers with 0-12 months of tenure have an observed churn rate of {early_tenure_churn_rate:.2f}%.",
        f"The selected month-to-month, high-charge, early-tenure segment has an observed churn rate of {segment_churn_rate:.2f}%."
    ]
})

insights

,Area,Finding
0,Overall Churn,Overall churn rate is 26.54%.
1,Contract,Month-to-month has the highest observed churn ...
2,Tenure,0-12 Months has the highest observed churn rat...
3,Monthly Charges,$60-$89 has the highest observed churn rate am...
4,Internet Service,Fiber optic has the highest observed churn rat...
5,Payment Method,Churn rates vary across payment-method categor...
6,Early Tenure,Customers with 0-12 months of tenure have an o...
7,Priority Segment,"The selected month-to-month, high-charge, earl..."


In [34]:
insights.to_csv(
    "../reports/telco_churn_insights.csv",
    index=False
)

In [35]:
kpi_summary.to_csv(
    "../reports/telco_churn_kpis.csv",
    index=False
)

In [36]:
contract_analysis.to_csv(
    "../reports/churn_by_contract.csv",
    index=False
)

tenure_analysis.to_csv(
    "../reports/churn_by_tenure.csv",
    index=False
)

charge_analysis.to_csv(
    "../reports/churn_by_monthly_charges.csv",
    index=False
)

internet_analysis.to_csv(
    "../reports/churn_by_internet_service.csv",
    index=False
)

In [37]:
print("==============================================")
print("TELCO CUSTOMER CHURN — EXECUTIVE SUMMARY")
print("==============================================")

print(f"""
The dataset contains {total_customers:,} customers, of which
{churned_customers:,} are recorded as churned and
{retained_customers:,} as retained.

The overall observed churn rate is {churn_rate:.2f}%.

The highest observed churn rate among contract categories is
associated with {highest_contract['Contract']} contracts,
at {highest_contract['Churn_Rate']:.2f}%.

The highest observed churn rate among tenure groups is
associated with {highest_tenure['Tenure_Group']},
at {highest_tenure['Churn_Rate']:.2f}%.

Customers with 0-12 months of tenure have an observed churn
rate of {early_tenure_churn_rate:.2f}%.

The {highest_charge_segment['Monthly_Charges_Group']} monthly-charge
group has the highest observed churn rate among the charge groups,
at {highest_charge_segment['Churn_Rate']:.2f}%.

These findings identify customer segments that can be examined
further for retention analysis. The relationships identified
in this dataset are associations and should not be interpreted
as proof of causation.
""")

TELCO CUSTOMER CHURN — EXECUTIVE SUMMARY

The dataset contains 7,043 customers, of which
1,869 are recorded as churned and
5,174 as retained.

The overall observed churn rate is 26.54%.

The highest observed churn rate among contract categories is
associated with Month-to-month contracts,
at 42.71%.

The highest observed churn rate among tenure groups is
associated with 0-12 Months,
at 47.44%.

Customers with 0-12 months of tenure have an observed churn
rate of 47.44%.

The $60-$89 monthly-charge
group has the highest observed churn rate among the charge groups,
at 33.74%.

These findings identify customer segments that can be examined
further for retention analysis. The relationships identified
in this dataset are associations and should not be interpreted
as proof of causation.

